# Session 07 Lab: Model Evaluation
**Course:** Noob to AI Expert | **Track:** Intermediate

Build a spam classifier and evaluate it with a full suite of metrics: accuracy, precision, recall, F1, AUC-ROC, and confusion matrix.

In [ ]:
!pip install scikit-learn matplotlib numpy -q
print('✅ Ready')

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, roc_auc_score, roc_curve)
import matplotlib.pyplot as plt
import numpy as np

# Use 20newsgroups as a binary classification proxy (rec.sport vs sci.med)
cats = ['rec.sport.hockey', 'sci.med']
train = fetch_20newsgroups(subset='train', categories=cats, remove=('headers', 'footers', 'quotes'))
test = fetch_20newsgroups(subset='test', categories=cats, remove=('headers', 'footers', 'quotes'))

vec = TfidfVectorizer(max_features=10000, stop_words='english')
X_train = vec.fit_transform(train.data)
X_test = vec.transform(test.data)
y_train, y_test = train.target, test.target

print(f'Train: {len(y_train)} | Test: {len(y_test)}')

In [ ]:
model = MultinomialNB()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print('=== Evaluation Metrics ===')
print(f'Accuracy:  {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred):.4f}')
print(f'F1 Score:  {f1_score(y_test, y_pred):.4f}')
print(f'AUC-ROC:   {roc_auc_score(y_test, y_prob):.4f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
im = ax1.imshow(cm, cmap='Blues')
ax1.set_xticks([0,1]); ax1.set_yticks([0,1])
ax1.set_xticklabels(cats, rotation=15); ax1.set_yticklabels(cats)
ax1.set_xlabel('Predicted'); ax1.set_ylabel('Actual')
ax1.set_title('Confusion Matrix')
for i in range(2):
    for j in range(2):
        ax1.text(j, i, cm[i,j], ha='center', va='center', fontsize=14, fontweight='bold')

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
ax2.plot(fpr, tpr, color='#3b82f6', lw=2, label=f'AUC={roc_auc_score(y_test, y_prob):.3f}')
ax2.plot([0,1],[0,1], 'k--', alpha=0.4)
ax2.set_xlabel('False Positive Rate'); ax2.set_ylabel('True Positive Rate')
ax2.set_title('ROC Curve')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()